Objective:

Build a basic chatbot that provides supportive and empathetic responses for stress, anxiety,
and emotional wellness.

Install important Libraries

In [ ]:
!pip install -q transformers[torch] datasets accelerate peft bitsandbytes

Load Dataset

In [ ]:
from datasets import load_dataset

# Loading the Parquet version of the dataset (the new standard)
dataset = load_dataset("Ahren09/empathetic_dialogues")

# Inspect the dataset structure
print(f"Dataset Structure: {dataset}")

# Let's show a sample to confirm the content
sample = dataset['train'][0]
print("\n--- Sample Empathetic Dialogue ---")
print(f"Context/Situation: {sample['prompt']}")
print(f"Empathetic Response: {sample['utterance']}")

In [ ]:
# 1. Fix the library conflict
!pip install --upgrade sympy typing-extensions

# 2. Re-install transformers to ensure everything is aligned
!pip install -U transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

# 1. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Set the padding token
# DistilGPT2 doesn't have a default pad token, so we use the end-of-text token
tokenizer.pad_token = tokenizer.eos_token

# 3. Load the Base Model
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Model {model_name} and Tokenizer loaded successfully!")

In [ ]:
def preprocess_function(examples):
    # Combine the prompt and the utterance into a single string for the model to learn from
    inputs = [f"Context: {p} Response: {u}" for p, u in zip(examples['prompt'], examples['utterance'])]

    # Tokenize the combined text
    # truncation=True ensures we don't exceed the model's memory limit
    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=128)

    # For Causal Language Modeling, the labels are the same as the input_ids
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# Apply the formatting to the whole dataset
# we use batched=True to make it much faster
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset['train'].column_names)

print("Dataset tokenized and formatted!")
print(f"Example of tokenized data: {tokenized_dataset['train'][0].keys()}")

In [ ]:
from transformers import TrainingArguments, Trainer

# 1. Define the training settings (Updated for transformers v4.46+)
training_args = TrainingArguments(
    output_dir="./empathetic-distilgpt2",
    eval_strategy="epoch",                # Fixed: renamed from evaluation_strategy
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    save_strategy="epoch",
    logging_steps=100,
    push_to_hub=False,
    report_to="none"
)

# 2. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)

# 3. Start the training!
print("Starting Fine-Tuning... Training on 84k rows typically takes 35-50 minutes on a T4.")
trainer.train()

In [ ]:
import os

# Define the permanent absolute path
final_save_path = "/content/empathetic-distilgpt2-final"

# Create the directory
if not os.path.exists(final_save_path):
    os.makedirs(final_save_path)

# Save the trained weights and the tokenizer
trainer.save_model(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"✅ Model safely locked in at: {final_save_path}")

In [ ]:
ls -l /content/empathetic-distilgpt2-final

In [ ]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer

# Replace these with your actual model/tokenizer if they are still in memory
# If they are gone, run the training cell first!
save_path = "/content/empathetic-distilgpt2-final"

if not os.path.exists(save_path):
    os.makedirs(save_path)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("✅ Weights saved to disk. They won't disappear now!")

In [ ]:
# 1. Save the model and tokenizer
model.save_pretrained("./empathetic-distilgpt2-final")
tokenizer.save_pretrained("./empathetic-distilgpt2-final")

# 2. Test the model with a stressful situation
def predict_empathy(user_input):
    # Format the input exactly like the training data
    input_text = f"Context: {user_input} Response:"

    # Tokenize and move to GPU
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    # Generate a response
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and clean up the text
    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
    # Return only the part after "Response:"
    return full_text.split("Response:")[-1].strip()

# Run a test
test_input = "I am feeling very overwhelmed with my work and I feel like I'm going to fail."
print(f"User: {test_input}")
print(f"Chatbot: {predict_empathy(test_input)}")

In [ ]:
def predict_empathy_v2(user_input):
    # Use a clearer separator
    input_text = f"Context: {user_input} Response:"

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        # Added repetition_penalty to stop it from echoing the user
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Logic to extract ONLY what comes after "Response:"
    if "Response:" in full_text:
        return full_text.split("Response:")[-1].strip()
    return full_text

# Run the test again
test_input = "I am feeling very overwhelmed with my work and I feel like I'm going to fail."
print(f"User: {test_input}")
print(f"Chatbot: {predict_empathy_v2(test_input)}")

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

# Set up the page
st.set_page_config(page_title="MindfulAI - Empathetic Support", page_icon="🌿")
st.title("🌿 MindfulAI Support")
st.markdown("A fine-tuned assistant for stress and emotional wellness.")

# Load model and tokenizer
@st.cache_resource
def load_model():
    model_path = "/content/empathetic-distilgpt2-final"
    if not os.path.exists(model_path):
        return None, None

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")
    return tokenizer, model

tokenizer, model = load_model()

# Initialize session state for chat
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if prompt := st.chat_input("How are you feeling today?"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        if model is not None:
            # 1. Clean Input
            priming_text = "I am so sorry to hear you're feeling that way, "
            input_text = f"User: {prompt}\nAssistant: {priming_text}"

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            # 2. Stable Generation Settings
            output_tokens = model.generate(
                **inputs,
                min_new_tokens=25,
                max_new_tokens=70,
                do_sample=True,
                temperature=0.6,
                top_k=40,
                top_p=0.9,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id
            )

            # 3. Cleanup & The "Hallucination Guard"
            full_response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

            # Split to get the Assistant's reply
            if "Assistant:" in full_response:
                response = full_response.split("Assistant:")[-1].strip().lstrip(',. ')
            else:
                response = full_response.split(prompt)[-1].strip().lstrip(',. ')

            # Fix dataset artifacts
            response = response.replace("_comma_", ",").replace("_", " ")

            # GUARDRULE: If the model hallucinates about kids/children, we fix it
            if any(word in response.lower() for word in ["kids", "children", "child"]):
                # Take the first sentence if it exists, otherwise use a default
                sentences = response.split(".")
                if len(sentences) > 0:
                    response = sentences[0] + ". I know it is hard, but I am here to listen if you want to talk more about it."
                else:
                    response = "I hope you can find a way to be kind to yourself today."

            # 4. Show the response
            st.markdown(response)
            st.session_state.messages.append({"role": "assistant", "content": response})
        else:
            st.error("Model failed to load. Please check your file path.")

In [ ]:
!pip install streamlit -q

In [ ]:
!ls -d /content/empathetic-distilgpt2-final

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
# 1. Kill everything
!pkill streamlit
!pkill npx

# 2. Start Streamlit FIRST and wait 5 seconds, then start the tunnel
!python -m streamlit run app.py --server.headless true & sleep 5 && (echo y | npx localtunnel --port 8501)